# 本轮离线分析伴随记录
从项目根目录运行前四个代码单元可复核关键汇总。第五单元是可选图像展示，尚未在本轮执行。原始逐帧计算在reprocess.py与analyze.py，不能将本笔记本汇总当成独立准确性验证。

In [1]:
from pathlib import Path
import json,csv,numpy as np
ROOT=Path.cwd()
BASE=ROOT/'04_实验设计/D435i_感知验证/TP-D435I-01/offline_reanalysis_v1'
OUT=BASE/'analysis_20261001_v1'
def readcsv(p):
    with p.open(encoding='utf-8-sig') as f:return list(csv.DictReader(f))
tr=readcsv(OUT/'trial_metrics.csv')
joints=readcsv(OUT/'joint_metrics.csv')
assert len(tr)==25 and len(joints)==175
print('trials:',len(tr),'joint rows:',len(joints))

trials: 25 joint rows: 175


In [2]:
n=sum(int(x['frames']) for x in tr)
v=sum(int(x['body_valid_frames']) for x in tr)
assert n==29938
print('dense body valid:',v,'/',n,'rate:',round(v/n,4))
print('per-task:',[(g['task_id'],round(g['dense_rate'],4),round(g['candidate_rate'],4)) for g in json.loads((OUT/'group_summary.json').read_text())])

dense body valid: 28369 / 29938 rate: 0.9476
per-task: [(1, 1.0, 0.1863), (2, 1.0, 0.6665), (3, 0.9651, 0.4296), (4, 0.9915, 0.7356), (5, 0.8778, 0.5952), (6, 0.9958, 0.6602), (7, 0.9197, 0.3664), (8, 0.7913, 0.0657)]


In [3]:
static=[x for x in joints if x['task_id'] in ['1','2'] and x['joint'] in ['r_shoulder','r_elbow','r_wrist']]
p95=np.array([float(x['position_p95_mm']) for x in static])
print('static P95 mm min/max:',p95.min(),p95.max())
for x in static[:3]:print(x['joint'],x['x_sd_mm'],x['y_sd_mm'],x['z_sd_mm'],x['u_sd_px'])

static P95 mm min/max: 4.309488035375514 36.261127508243355
r_shoulder 2.080461927075981 2.213363394496033 5.616151951733159 1.0006137422158021
r_elbow 12.57137910166656 2.3341401423246206 3.1362678290552077 8.60530221891648
r_wrist 6.476789576280483 1.2006713956308923 1.5426472571175969 4.569070579761218


In [4]:
for x in tr:
    if x['task_id']=='1':print(x['run_id'],x['original_alert_share_of_observable'],x['recentered_alert_share_of_observable'])
print('all final candidate counts:',sorted({x['final_reps'] for x in tr}))
print('Note: local baseline is counterfactual; instruction is not ground truth.')

eight_S1_20261001_131959_59d5ec91 0.0 0.0
eight_S1_20261001_132059_bc0d3f66 0.9933333333333333 0.0
eight_S1_20261001_132148_b0ac2f4c 0.9933333333333333 0.0
all final candidate counts: ['0']
Note: local baseline is counterfactual; instruction is not ground truth.


In [ ]:
from IPython.display import Image,display
display(Image(filename=str(OUT/'coverage_comparison.png')))
display(Image(filename=str(OUT/'static_stability.png')))